# Total Snippets Extraction
This notebook extract all opium mentions from the full library to prepare opium snippets for topic modelling (see 3_analysis).

In [1]:
import os
import re
import pandas as pd
import spacy
import ipywidgets as widgets
from tqdm.auto import tqdm

from IPython.display import display, HTML

# Load spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import spacy.cli
    spacy.cli.download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

In [2]:
# Setup Paths
metadata_path = '../../data/metadata_files/GP_opium_filtered_1850_1930.parquet'
fulltext_path = '../../opium_books_fulltext'

output_parquet = '../../data/snippets/total_snippets.parquet'
output_csv = '../../data/snippets/total_snippets.csv'

window_size = 100

## 1. Extracting context windows out of all books

In [24]:
books = pd.read_parquet(metadata_path)
books["Etext Number"]

0          15
1          16
3          27
6          44
7          60
        ...  
6396    74956
6403    75246
6406    75372
6407    75497
6408    75518
Name: Etext Number, Length: 4378, dtype: int64

In [28]:
keywords = books['Opium Keywords'].explode().unique().tolist()
keywords

['opium',
 'paregoric',
 'laudanum',
 'heroin',
 'narcotic',
 'morphine',
 'anodyne',
 'soporific',
 'nepenthe',
 'chandu',
 'codein',
 'dover’s powder']

In [36]:
results = []
if os.path.exists(fulltext_path):
    for book_id in book_ids:
        filepath = os.path.join(fulltext_path, str(book_id))
        if os.path.isfile(filepath):
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
                text_lower = text.lower()
                for keyword in keywords:
                    pattern = r'\b' + re.escape(keyword) + r'\b'
                    for match in re.finditer(pattern, text_lower):
                        idx = match.start()
                        left_text = text[:idx]
                        right_text = text[idx + len(keyword):]
                        
                        left_words = left_text.split()
                        right_words = right_text.split()
                        
                        context_left = ' '.join(left_words[-window_size:])
                        context_right = ' '.join(right_words[:window_size])
                        
                        results.append({
                            'Book_ID': book_id,
                            'Keyword': keyword,
                            'Snippet': f"{context_left} {text[idx:idx+len(keyword)]} {context_right}"
                        })
                        
df = pd.DataFrame(results)
print(f"Found {len(df)} keyword mentions across all {len(book_ids)} books.")


Found 6658 keyword mentions across all 4378 books.


Also we remove overlapping snippets in 2 ways:
1. Within-book: if two snippets from same book + same keyword are too similar OR one contains the other → drop one
2. Cross-book: detect edition duplicates (text similarity, high threshold) → drop one

In [43]:
def normalize(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def deduplicate_snippets(df,
                         within_book_sim=0.85,
                         cross_book_sim=0.95):

    print(f"Original snippets: {len(df)}")

    df = df.copy()
    df['norm'] = df['Snippet'].map(normalize)

    to_drop = set()

    # ---------------------------
    # 1. WITHIN-BOOK DEDUP
    # ---------------------------
    for book_id, group in df.groupby('Book_ID'):

        idxs = group.index.tolist()

        for i in range(len(idxs)):
            for j in range(i + 1, len(idxs)):

                a = idxs[i]
                b = idxs[j]

                if a in to_drop or b in to_drop:
                    continue

                s1 = df.at[a, 'norm']
                s2 = df.at[b, 'norm']

                # containment-aware similarity (good for overlapping windows)
                sim = fuzz.token_set_ratio(s1, s2) / 100

                if sim >= within_book_sim:
                    # drop shorter / redundant one
                    if len(s1) <= len(s2):
                        to_drop.add(a)
                    else:
                        to_drop.add(b)

    # ---------------------------
    # 2. CROSS-BOOK DEDUP (EDITION MATCHING)
    # ---------------------------
    remaining = df.drop(index=list(to_drop)).copy()
    remaining_idxs = remaining.index.tolist()

    for i in range(len(remaining_idxs)):
        for j in range(i + 1, len(remaining_idxs)):

            a = remaining_idxs[i]
            b = remaining_idxs[j]

            if a in to_drop or b in to_drop:
                continue

            # skip same book here (already handled above)
            if remaining.at[a, 'Book_ID'] == remaining.at[b, 'Book_ID']:
                continue

            s1 = remaining.at[a, 'norm']
            s2 = remaining.at[b, 'norm']

            sim = fuzz.token_set_ratio(s1, s2) / 100

            if sim >= cross_book_sim:
                # treat as duplicate edition snippet
                to_drop.add(b)

    # final output
    df_dedup = df.drop(index=list(to_drop)).drop(columns=['norm'])

    print(f"Dropped {len(to_drop)} snippets")
    print(f"Remaining snippets: {len(df_dedup)}")

    return df_dedup

In [44]:
df_dedup = deduplicate_snippets(df)

Original snippets: 6658
Dropped 1288 snippets
Remaining snippets: 5370


In [46]:
df_dedup[['Book_ID', 'Keyword', 'Snippet']].to_csv(output_csv)
df_dedup[['Book_ID', 'Keyword', 'Snippet']].to_parquet(output_parquet)

## 2. Manual Inspection Viewer
Use the slider to browse through the extracted snippets. The target keyword is highlighted.

In [ ]:
pd.set_option('display.max_colwidth', None)
def view_snippet(index):
    if len(df_dedup) == 0:
        print("No snippets found.")
        return
    row = df_dedup.iloc[index]
    html_out = f"""
    <div style='font-family: Georgia, serif; font-size: 16px; line-height: 1.6; max-width: 800px; padding: 20px; border: 1px solid #ccc; border-radius: 5px; background: #f9f9f9;'>
        <h4>Book ID: {row['Book_ID']} | Keyword: <span style='color: dimgrey;'>{row['Keyword'].upper()}</span></h4>
        <hr>
        <p>
            {row['Left_Context']} 
            <span style='background-color: #ffeb3b; font-weight: bold; padding: 0 4px;'>{row['Keyword']}</span> 
            {row['Right_Context']}
        </p>
    </div>
    """
    display(HTML(html_out))

In [ ]:
if len(df) > 0:
    slider = widgets.IntSlider(min=0, max=len(df_dedup)-1, step=1, description='Snippet:', layout=widgets.Layout(width='800px'))
    widgets.interact(view_snippet, index=slider)

interactive(children=(IntSlider(value=0, description='Snippet:', layout=Layout(width='800px'), max=3856), Outp…

## 3. Authors extraction

In [3]:
all_snippets_path = "../../data/snippets/total_snippets.csv"
df_snippets = pd.read_csv(all_snippets_path, index_col=0).rename(columns={"Book_ID":"Etext Number"})

books_metadata_path = "../../data/metadata_files/metadata_1870_1920_with_gender.csv"
df_metadata = pd.read_csv(books_metadata_path, index_col=0)

df_snippets = df_snippets.merge(df_metadata, on="Etext Number")
df_snippets = df_snippets[~df_snippets["Keyword"].isin(["poppy", "black smoke"])]
df_snippets

,Etext Number,Keyword,Snippet,Title,Authors,LoCC,Bookshelves,Subjects,rights,Published Year,Normalised Authors,Author Gender,Author Nationality,Author Birth,Author Death,Author Info Source,Author Info Found,Number of books by author
0,15,opium,they would rather not see whales than otherwis...,"Moby-Dick; or, The Whale","Melville, Herman",PS,Adventure; Best Books Ever Listings; Browsing:...,"Whales -- Fiction ; Ahab, Captain (Fictitious ...",Public domain in the USA.,1851,Herman Melville,male,American,1819.0,1891.0,Wikipedia,True,1
1,15,paregoric,questionable; for it is not customary for such...,"Moby-Dick; or, The Whale","Melville, Herman",PS,Adventure; Best Books Ever Listings; Browsing:...,"Whales -- Fiction ; Ahab, Captain (Fictitious ...",Public domain in the USA.,1851,Herman Melville,male,American,1819.0,1891.0,Wikipedia,True,1
2,15,laudanum,previous when the Pequod spoke the Town-Ho. Ac...,"Moby-Dick; or, The Whale","Melville, Herman",PS,Adventure; Best Books Ever Listings; Browsing:...,"Whales -- Fiction ; Ahab, Captain (Fictitious ...",Public domain in the USA.,1851,Herman Melville,male,American,1819.0,1891.0,Wikipedia,True,1
3,44,opium,and scared you! Nothing to cry about. I’m the ...,The Song of the Lark,"Cather, Willa",PS,Opera; Browsing: Culture/Civilization/Society;...,Opera -- Fiction ; Chicago (Ill.) -- Fiction ;...,Public domain in the USA.,1915,Willa Cather,female,American,1873.0,1947.0,Wikipedia,True,7
4,95,narcotic,"that?” he asked. “Why this,” I answered. “That...",The prisoner of Zenda,"Hope, Anthony",PR,Adventure; Best Books Ever Listings; Movie Boo...,Kings and rulers -- Fiction ; Love stories ; I...,Public domain in the USA.,1894,Anthony Hope,male,"English, British",1863.0,1933.0,Wikipedia,True,31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4669,74238,opium,"going to defend, only to excuse him a little. ...","Illustrations of political economy, Volume 8 (...","Martineau, Harriet",PR,Browsing: Economics; Browsing: Fiction,"Political fiction, English ; Social problems -...",Public domain in the USA.,1923,Harriet Martineau,female,English,1802.0,1876.0,Wikipedia,True,1
4670,74270,opium,"Then he bent his face to the table, and his fo...","A woman's trust; or, Lady Elaine's martyrdom","Clay, Bertha M.",PS,Browsing: Fiction,England -- Fiction ; PS ; Dime novels ; Popula...,Public domain in the USA.,1883,Bertha M. Clay,NaN,NaN,1836.0,1884.0,OpenLibrary,True,3
4671,75518,opium,"then, but not until formally invited, fills hi...",The just steward,"Dehan, Richard",PR,NaN,NaN,Public domain in the USA.,1922,Richard Dehan,female,Irish,1863.0,1932.0,"Wikipedia, OpenLibrary, Wikidata",True,5
4672,75518,opium,"Shaykh, sending out a volume of cigarette-smok...",The just steward,"Dehan, Richard",PR,NaN,NaN,Public domain in the USA.,1922,Richard Dehan,female,Irish,1863.0,1932.0,"Wikipedia, OpenLibrary, Wikidata",True,5


In [4]:
summary = pd.DataFrame({
    "nb_opium_books": df_snippets.groupby("Authors")["Etext Number"].nunique(),
    "total_nb_snippets": df_snippets.groupby("Authors").size(),
})

snippets_per_book = (
    df_snippets
    .groupby(["Authors", "Etext Number"])
    .size()
    .reset_index(name="snippet_count")
)

# Add book titles
book_titles = (
    df_metadata[["Etext Number", "Title"]]
    .drop_duplicates("Etext Number")
)

snippets_per_book = snippets_per_book.merge(
    book_titles,
    on="Etext Number",
    how="left"
)

summary["snippets_per_book"] = (
    snippets_per_book
    .groupby("Authors")
    .apply(
        lambda g: {
            (row["Etext Number"], row["Title"]): row["snippet_count"]
            for _, row in g.iterrows()
        },
        include_groups=False
    )
)

summary["max_nb_snippets_per_book"] = (
    summary["snippets_per_book"].apply(lambda d: max(d.values()))
)

summary = summary.sort_values("total_nb_snippets", ascending=False)

sub_df = (
    df_metadata[["Authors", "Author Gender", "Author Nationality", "Number of books by author"]]
    .drop_duplicates()
)

summary = summary.merge(sub_df, on="Authors")

summary

,Authors,nb_opium_books,total_nb_snippets,snippets_per_book,max_nb_snippets_per_book,Author Gender,Author Nationality,Number of books by author
0,"Rohmer, Sax",11,183,"{(173, 'The Insidious Dr. Fu Manchu'): 22, (11...",66,male,English,13
1,"Collins, Wilkie",12,95,"{(155, 'The Moonstone'): 60, (1438, 'No Name')...",60,female,English,15
2,"Ottolengui, Rodrigues",1,92,"{(32985, 'A Modern Wizard'): 92}",92,male,American,2
3,"Reeve, Arthur B. (Arthur Benjamin)",13,80,"{(2454, 'The Silent Bullet'): 10, (5007, 'The ...",20,male,American,14
4,"Roe, Edward Payson",4,68,"{(5433, 'Without a Home'): 64, (6090, 'What Ca...",64,male,American,14
...,...,...,...,...,...,...,...,...
847,"Lauder, Thomas Dick, Sir",1,1,"{(58931, 'Legendary Tales of the Highlands (Vo...",1,male,Scottish,1
848,"Lane, Elinor Macartney",1,1,"{(28366, 'Nancy Stair: A Novel'): 1}",1,female,American,1
849,"Kummer, Frederic Arnold",1,1,"{(33019, 'The Green God'): 1}",1,male,American,4
850,"Kirkman, Marshall M. (Marshall Monroe)",1,1,"{(53006, 'The Romance of Gilbert Holmes: An Hi...",1,male,American,1


In [ ]:
path_to_opium_authors = "../../data/metadata_files/opium_authors_snippet_nb.csv"
#summary.to_csv(path_to_opium_authors)

### Cases studies

#### Sax Rohmer, Dope (1919) --> male gaze

In [12]:
df_snippets[df_snippets["Authors"]=="Rohmer, Sax"]

,Etext Number,Keyword,Snippet,Title,Authors,LoCC,Bookshelves,Subjects,rights,Published Year,Normalised Authors,Author Gender,Author Nationality,Author Birth,Author Death,Author Info Source,Author Info Found,Number of books by author
91,173,opium,"afraid to trust you--yet. Be comforted, for th...",The Insidious Dr. Fu Manchu,"Rohmer, Sax",PR,Crime Fiction; Browsing: Crime/Mystery; Browsi...,Criminals -- Fiction ; Detective and mystery s...,Public domain in the USA.,1913,Sax Rohmer,male,English,1883.0,1959.0,Wikipedia,True,13
92,173,opium,'lascar' as the dacoit who was murdered by Fu-...,The Insidious Dr. Fu Manchu,"Rohmer, Sax",PR,Crime Fiction; Browsing: Crime/Mystery; Browsi...,Criminals -- Fiction ; Detective and mystery s...,Public domain in the USA.,1913,Sax Rohmer,male,English,1883.0,1959.0,Wikipedia,True,13
93,173,opium,"""You are forgetting me, Smith,"" I said. He tur...",The Insidious Dr. Fu Manchu,"Rohmer, Sax",PR,Crime Fiction; Browsing: Crime/Mystery; Browsi...,Criminals -- Fiction ; Detective and mystery s...,Public domain in the USA.,1913,Sax Rohmer,male,English,1883.0,1959.0,Wikipedia,True,13
94,173,opium,could get through to the front and watch from ...,The Insidious Dr. Fu Manchu,"Rohmer, Sax",PR,Crime Fiction; Browsing: Crime/Mystery; Browsi...,Criminals -- Fiction ; Detective and mystery s...,Public domain in the USA.,1913,Sax Rohmer,male,English,1883.0,1959.0,Wikipedia,True,13
95,173,opium,"I was with this form of gentle persuasion. ""Ko...",The Insidious Dr. Fu Manchu,"Rohmer, Sax",PR,Crime Fiction; Browsing: Crime/Mystery; Browsi...,Criminals -- Fiction ; Detective and mystery s...,Public domain in the USA.,1913,Sax Rohmer,male,English,1883.0,1959.0,Wikipedia,True,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2286,19706,opium,"the finish, for Robert had said, ""I shan't run...",Brood of the Witch-Queen,"Rohmer, Sax",PR,Crime Fiction; Browsing: Crime/Mystery; Browsi...,Witches -- Fiction ; PR,Public domain in the USA.,1918,Sax Rohmer,male,English,1883.0,1959.0,Wikipedia,True,13
2563,27461,opium,"of very extraordinary talent,"" said Thessaly, ...",The Orchard of Tears,"Rohmer, Sax",PR,Browsing: Literature; Browsing: Fiction,Man-woman relationships -- Fiction ; PR,Public domain in the USA.,1918,Sax Rohmer,male,English,1883.0,1959.0,Wikipedia,True,13
2564,27461,opium,"excursion, one night visiting a private gaming...",The Orchard of Tears,"Rohmer, Sax",PR,Browsing: Literature; Browsing: Fiction,Man-woman relationships -- Fiction ; PR,Public domain in the USA.,1918,Sax Rohmer,male,English,1883.0,1959.0,Wikipedia,True,13
2565,27461,opium,he had pierced to the heart of the labyrinth h...,The Orchard of Tears,"Rohmer, Sax",PR,Browsing: Literature; Browsing: Fiction,Man-woman relationships -- Fiction ; PR,Public domain in the USA.,1918,Sax Rohmer,male,English,1883.0,1959.0,Wikipedia,True,13


In [13]:
df_snippets[df_snippets["Authors"]=="Rohmer, Sax"].to_csv("../../data/case_studies/SaxRohmer_snippets.csv")

#### Myrtle Reed, A Spinner in the Sun (1911) --> american female author

In [14]:
df_snippets[df_snippets["Authors"]=="Reed, Myrtle"]

,Etext Number,Keyword,Snippet,Title,Authors,LoCC,Bookshelves,Subjects,rights,Published Year,Normalised Authors,Author Gender,Author Nationality,Author Birth,Author Death,Author Info Source,Author Info Found,Number of books by author
1823,12672,laudanum,"set aside. Every one came to this, sooner or l...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1824,12672,laudanum,"Miss Hitty, tasted of the soup. A little later...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1825,12672,laudanum,"crossed the tracks again, at the deserted poin...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1826,12672,laudanum,"side of the house was, as yet, untouched, and ...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1827,12672,laudanum,about among the rubbish. By a flash of intuiti...,A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1828,12672,laudanum,"shall clasp thee again, And with God be the re...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1829,12672,laudanum,"easier. He rapped once, with hesitation, then ...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1830,12672,laudanum,"you. God knows I haven't forgiven him myself, ...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1831,12672,anodyne,"She mused, ironically, upon the permanence of ...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1832,12672,soporific,a great black poppy with a deadly fragrance sp...,A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9


In [15]:
df_snippets[df_snippets["Authors"]=="Reed, Myrtle"].to_csv("../../data/case_studies/MyrtleReed_snippets.csv")

#### George Eliot, Middlemarch (1871) --> british female author

In [16]:
df_snippets[df_snippets["Authors"]=="Eliot, George"]

,Etext Number,Keyword,Snippet,Title,Authors,LoCC,Bookshelves,Subjects,rights,Published Year,Normalised Authors,Author Gender,Author Nationality,Author Birth,Author Death,Author Info Source,Author Info Found,Number of books by author
16,145,opium,it may confidently await those messages from t...,Middlemarch,"Eliot, George",PR,Historical Fiction; Best Books Ever Listings; ...,City and town life -- Fiction ; Love stories ;...,Public domain in the USA.,1871,George Eliot,female,English,1819.0,1880.0,Wikipedia,True,9
17,145,opium,guinea when everybody knows that the seats hav...,Middlemarch,"Eliot, George",PR,Historical Fiction; Best Books Ever Listings; ...,City and town life -- Fiction ; Love stories ;...,Public domain in the USA.,1871,George Eliot,female,English,1819.0,1880.0,Wikipedia,True,9
18,145,opium,of miracle-workers. Some of that twice-blessed...,Middlemarch,"Eliot, George",PR,Historical Fiction; Best Books Ever Listings; ...,City and town life -- Fiction ; Love stories ;...,Public domain in the USA.,1871,George Eliot,female,English,1819.0,1880.0,Wikipedia,True,9
19,145,opium,"of gambling in Paris, watching it as if it had...",Middlemarch,"Eliot, George",PR,Historical Fiction; Best Books Ever Listings; ...,City and town life -- Fiction ; Love stories ;...,Public domain in the USA.,1871,George Eliot,female,English,1819.0,1880.0,Wikipedia,True,9
20,145,opium,"disease. Still, new symptoms may arise. I shal...",Middlemarch,"Eliot, George",PR,Historical Fiction; Best Books Ever Listings; ...,City and town life -- Fiction ; Love stories ;...,Public domain in the USA.,1871,George Eliot,female,English,1819.0,1880.0,Wikipedia,True,9
21,145,opium,"here yourself?” said Lydgate, looking at Bulst...",Middlemarch,"Eliot, George",PR,Historical Fiction; Best Books Ever Listings; ...,City and town life -- Fiction ; Love stories ;...,Public domain in the USA.,1871,George Eliot,female,English,1819.0,1880.0,Wikipedia,True,9
22,145,opium,"at the persistent life in this man, whom he wo...",Middlemarch,"Eliot, George",PR,Historical Fiction; Best Books Ever Listings; ...,City and town life -- Fiction ; Love stories ;...,Public domain in the USA.,1871,George Eliot,female,English,1819.0,1880.0,Wikipedia,True,9
23,145,opium,Bulstrode began to administer the opium accord...,Middlemarch,"Eliot, George",PR,Historical Fiction; Best Books Ever Listings; ...,City and town life -- Fiction ; Love stories ;...,Public domain in the USA.,1871,George Eliot,female,English,1819.0,1880.0,Wikipedia,True,9
24,145,opium,"him from seeing the one probability to be, tha...",Middlemarch,"Eliot, George",PR,Historical Fiction; Best Books Ever Listings; ...,City and town life -- Fiction ; Love stories ;...,Public domain in the USA.,1871,George Eliot,female,English,1819.0,1880.0,Wikipedia,True,9
25,145,opium,"towards Raffles’s room, and he could hear him ...",Middlemarch,"Eliot, George",PR,Historical Fiction; Best Books Ever Listings; ...,City and town life -- Fiction ; Love stories ;...,Public domain in the USA.,1871,George Eliot,female,English,1819.0,1880.0,Wikipedia,True,9


In [17]:
df_snippets[df_snippets["Authors"]=="Eliot, George"].to_csv("../../data/case_studies/GeorgeEliot_snippets.csv")